# 기본 확인

metadata 로드 - 전처리(시간,'Capacity','Re','Rct'수치형 변환) - 결측치 확인 - 시간 확인

데이터 나눔처리 ('B0005', 'B0006', 'B0007', 'B0018' 배터리 대상으로 충전 / 방전 / 임피던스별 data) 

# A
# A
# A
# A
# A
# A



# RSH 낙서장입니다



# B
# B
# B
# B
# B
# B



In [ ]:
import re
import pandas as pd
import numpy as np

# [함수 1] 지수 표기법 대응 파싱 (보정/보간 없음)
def perfect_parse_time(x):
    if pd.isna(x): return pd.NaT
    try:
        s = str(x)
        # 지수 표기법 덩어리를 하나로 인식하는 정규식
        pattern = r"[-+]?\d*\.?\d+[eE][-+]?\d+|[-+]?\d+\.?\d*"
        parts = re.findall(pattern, s)
        
        if not parts: return pd.NaT
        
        # float 변환 후 정수화
        nums = [int(round(float(p))) for p in parts]
        
        if len(nums) >= 3:
            # 추출된 숫자 그대로 Timestamp 생성 시도
            # (만약 여기서 nums[0]이 2라면 0002년으로 생성됨 -> 에러 안남)
            # (단, 월이 0이거나 일이 32인 경우 등은 여기서 에러가 나서 NaT로 감)
            return pd.Timestamp(year=nums[0], month=nums[1], day=nums[2], 
                                hour=nums[3] if len(nums)>3 else 0,
                                minute=nums[4] if len(nums)>4 else 0,
                                second=nums[5] if len(nums)>5 else 0)
    except:
        pass
    return pd.NaT

# [함수 2] 수치형 데이터 정제
def clean_numeric(x):
    if pd.isna(x): return np.nan
    val = str(x).replace('[', '').replace(']', '').strip()
    try:
        return float(val)
    except:
        return np.nan

# --- [실행 및 검증 섹션] ---
df = df_meta.copy()

# 1. 파싱 적용
df['parsed_time'] = df['start_time'].apply(perfect_parse_time)

# 2. 수치형 정제
for col in ['Capacity', 'Re', 'Rct']:
    if col in df.columns:
        df[col] = df[col].apply(clean_numeric)

# 3. 결측치(NaT)인 데이터만 추출
# (사용자님이 말씀하신 '문제가 되는 46개'가 여기 담깁니다)
failed_cases = df[df['parsed_time'].isna()].copy()

print("-" * 60)
print(f"✅ 전체 데이터 중 파싱 실패(NaT) 개수: {len(failed_cases)}개")
print("-" * 60)

# 4. 실패한 데이터 46개의 원본 값 출력
if not failed_cases.empty:
    print(f"{'Index':<8} | {'Battery_ID':<12} | {'Original start_time'}")
    print("-" * 60)
    for idx, row in failed_cases.iterrows():
        print(f"{idx:<8} | {str(row['battery_id']):<12} | {row['start_time']}")
else:
    print("🎉 파싱에 실패한 데이터가 없습니다!")

# 5. (참고) 연도가 2000년 미만으로 나온 '이상한' 데이터가 있는지 추가 확인
# (에러는 안 났지만 연도가 잘못 찍힌 케이스 확인용)
weird_years = df[(df['parsed_time'].notna()) & (df['parsed_time'].dt.year < 2000)]
if not weird_years.empty:
    print(f"\n⚠️ 연도는 추출됐으나 2000년 미만인 데이터: {len(weird_years)}개")
    print(weird_years[['battery_id', 'start_time', 'parsed_time']].head())

In [1]:
# 도구 불러오기
import pandas as pd
import numpy as np
import glob
import os
import re
import ast
from glob import glob
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import platform
import matplotlib.font_manager as fm

# 판다스 출력 제한 해제 
pd.set_option('display.max_rows', 100) # 최대 100행까지 생략 없이 출력
pd.set_option('display.max_columns', None) 
pd.set_option('display.width', 1000)

In [2]:
# 원본 메타데이터 로드
df_meta = pd.read_csv("../../dataset/metadata.csv")

In [ ]:
print(df_meta['battery_id'].value_counts())
df_meta

In [ ]:
# 데이터 확인 (null/Dtype)
df_meta.info()

#-> 시간 object & Capacity,Re,Rct objct & Capacity,Re,Rct null 확인

In [ ]:
df_meta['start_time'].describe()

In [3]:
# 전처리 (시간/수치형(Capacity, Re, Rct)/file_num 파생)

def clean_parse_time(x):
    if pd.isna(x):
        return pd.NaT
    
    try:
        s = str(x)
        pattern = r"[-+]?\d*\.?\d+[eE][-+]?\d+|[-+]?\d+\.?\d*"
        parts = re.findall(pattern, s)
        
        if len(parts) < 3:
            return pd.NaT
        
        nums = [float(p) for p in parts]
        
        year   = int(nums[0])
        month  = int(nums[1])
        day    = int(nums[2])
        hour   = int(nums[3]) if len(nums) > 3 else 0
        minute = int(nums[4]) if len(nums) > 4 else 0
        
        # 핵심: round 제거
        second = int(nums[5]) if len(nums) > 5 else 0
        
        return pd.Timestamp(year=year, month=month, day=day,
                            hour=hour, minute=minute, second=second)
    
    except Exception as e:
        print(f"[파싱 실패] {x} | {e}")
        return pd.NaT

def clean_numeric(x):
    """수치형 데이터의 대괄호 제거 및 float 변환"""
    if pd.isna(x): return np.nan
    val = str(x).replace('[', '').replace(']', '').strip()
    try:
        return float(val)
    except:
        return np.nan

# --- [전처리 실행] ---
df = df_meta.copy()

# 1. 시간 데이터 정제 (소수점 초까지 완벽 대응)
df['start_time'] = df['start_time'].apply(clean_parse_time)

# 2. 수치형 데이터 정제 (Capacity, Re, Rct)
for col in ['Capacity', 'Re', 'Rct']:
    if col in df.columns:
        df[col] = df[col].apply(clean_numeric)

# 3. 물리적 순서 정렬 (파일 번호 기준)
df['file_num'] = df['filename'].str.extract(r'(\d+)').astype(int)
df = df.sort_values(['battery_id', 'file_num']).reset_index(drop=True)

# 최종 결과 검증
print(f"전체 데이터 개수: {len(df):,}개")
print(f"시간 결측치(NaT): {df['start_time'].isna().sum()}개")
print(f"수치형 결측치(Cap): {df['Capacity'].isna().sum()}개")


전체 데이터 개수: 7,565개
시간 결측치(NaT): 0개
수치형 결측치(Cap): 4796개


In [ ]:
df['start_time'].describe()

In [ ]:
df

In [ ]:
# 날짜별 B005의 방전데이터 전압확인

# 1. B0005 방전 데이터 필터링 및 시간순 정렬
b0005_discharge = df[(df['battery_id'] == 'B0005') & (df['type'] == 'discharge')].copy()
b0005_discharge['start_time'] = pd.to_datetime(b0005_discharge['start_time'])
b0005_discharge = b0005_discharge.sort_values(by='start_time').reset_index(drop=True)

# 2. 이중 축 그래프 생성
fig, ax1 = plt.subplots(figsize=(14, 7))
sns.set_style("whitegrid")

# --- 왼쪽 축: Capacity (용량) ---
color_cap = 'tab:blue'
ax1.set_xlabel('Start Time', fontsize=12)
ax1.set_ylabel('Capacity (Ah)', color=color_cap, fontsize=12)
line1 = ax1.plot(b0005_discharge['start_time'], b0005_discharge['Capacity'], 
                 marker='o', markersize=3, label='Capacity', color=color_cap, linewidth=1.5)
ax1.tick_params(axis='y', labelcolor=color_cap)

# --- 오른쪽 축: Ambient Temperature (주변 온도) ---
ax2 = ax1.twinx() # x축을 공유하는 두 번째 y축 생성
color_temp = 'tab:red'
ax2.set_ylabel('Ambient Temperature (°C)', color=color_temp, fontsize=12)
line2 = ax2.plot(b0005_discharge['start_time'], b0005_discharge['ambient_temperature'], 
                 label='Ambient Temp', color=color_temp, linestyle='--', alpha=0.7)
ax2.tick_params(axis='y', labelcolor=color_temp)
ax2.set_ylim(0, 30) # 온도 변화가 미세할 경우 범위를 좁혀서 확인 가능

# 3. 그래프 꾸미기
plt.title('B0005 Discharge: Capacity & Temperature Trend over Time', fontsize=16, pad=20)
fig.tight_layout()

# 범례 합치기
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='center right')

plt.xticks(rotation=45)
plt.show()

In [ ]:
# 1. B0005 배터리의 'discharge' 데이터만 필터링
b0005_discharge = df[(df['battery_id'] == 'B0005') & (df['type'] == 'discharge')].copy()

# 2. start_time을 기준으로 정렬 (이미 되어 있는지 확인하기 위함)
b0005_discharge = b0005_discharge.sort_values(by='start_time')

# 3. 주요 컬럼(파일명, 시작시간, 용량) 출력
# 모든 행을 보고 싶다면 pd.set_option을 사용하세요.
pd.set_option('display.max_rows', None) # 모든 행 출력 설정

print(f"📊 [B0005] Discharge 데이터 시간순 목록 (총 {len(b0005_discharge)}개)")
print("-" * 60)
display(b0005_discharge)
print("-" * 60)

# 다시 기본 설정으로 복구 (옵션)
pd.reset_option('display.max_rows')

In [ ]:
# 카운트별 시간

# 1. B0005 방전 데이터 필터링 (필터링만 하고 정렬은 하지 않은 원본 상태 유지)
b0005_discharge = df[(df['battery_id'] == 'B0005') & (df['type'] == 'discharge')].copy()

# 2. start_time을 datetime으로 변환
b0005_discharge['start_time'] = pd.to_datetime(b0005_discharge['start_time'])

# 3. 데이터가 들어있는 순서대로 인덱스 부여 (카운트)
b0005_discharge['data_count'] = range(1, len(b0005_discharge) + 1)

# 4. 시각화
plt.figure(figsize=(12, 6))
sns.set_style("whitegrid")

# 데이터 카운트별 시간 흐름 그래프
plt.plot(b0005_discharge['data_count'], b0005_discharge['start_time'], 
         marker='o', markersize=3, linestyle='-', color='darkcyan', label='Start Time Flow')

# 그래프 꾸미기
plt.title('B0005 Discharge: Data Order vs. Actual Time Flow', fontsize=15, pad=20)
plt.xlabel('Data Count (Row Order in DF)', fontsize=12)
plt.ylabel('Actual Start Time', fontsize=12)

# y축 시간이 겹치지 않게 포맷 조정
plt.gcf().autofmt_xdate() 

plt.tight_layout()
plt.show()

In [ ]:
# 데이터 시간 범위 확인
# 1. 전체 시간 범위 확인 (Min, Max)
start_date = df['start_time'].min()
end_date = df['start_time'].max()
duration = end_date - start_date

print("=== [데이터셋 시간 범위 결과] ===")
print(f"전체 시작 시점: {start_date}")
print(f"전체 종료 시점: {end_date}")
print(f"총 실험 기간  : {duration.days}일 {duration.seconds // 3600}시간")

# 2. 배터리별 시간 범위 확인 (각 배터리마다 실험 시점이 다를 수 있음)
print("\n=== [배터리 ID별 시간 범위] ===")
battery_time_range = df.groupby('battery_id')['start_time'].agg(['min', 'max', 'count', 'nunique'])
battery_time_range.columns = ['시작 시간', '종료 시간', '행 수', '파일 수']
display(battery_time_range)

# 전체 2008-04-02~2010-09-30 데이터였고,
# 해당 배터리는 2008-04-02~2008-05-28 (616사이클) & 07-07~08-20 (319사이클)이었음

In [4]:
# 'B0005', 'B0006', 'B0007', 'B0018' 배터리 대상으로 충전 / 방전 / 임피던스별 data 구성하기
# SOH EOL RUL cycle 파생

# 대상 배터리 ID 리스트
target_battery_ids = ['B0005', 'B0006', 'B0007', 'B0018']

# 데이터 저장할 폴더 경로
data_folder = "../../dataset/data"

# 데이터를 쌓아두기 위한 저장소(Dictionary) 생성
collected_data = {}


### 각 배터리별 EOL 사이클 번호를 먼저 파악 (RUL 계산용)
# SOH 80% 기준을 찾기 위해 메타데이터(df)에서 미리 계산
eol_dict = {}
for b_id in target_battery_ids:
    b_meta = df[(df['battery_id'] == b_id) & (df['type'] == 'discharge')].copy()
    if not b_meta.empty:
        initial_cap = b_meta['Capacity'].iloc[0] # 첫 번째 방전 용량
        # SOH 80% 이하인 첫 번째 행의 인덱스(순번) 찾기
        eol_idx = np.where((b_meta['Capacity'] / initial_cap) * 100 <= 80)[0]
        eol_dict[b_id] = eol_idx[0] + 1 if len(eol_idx) > 0 else np.nan

for battery_id in target_battery_ids:
    # 시간순 정렬 (매칭 오류 방지)
    filtered_df = df[df['battery_id'] == battery_id].sort_values('start_time')
    cycle_counters = {'charge': 1, 'discharge': 1, 'impedance': 1}
    
    # 기준 용량 정의 (해당 배터리의 첫 번째 방전 용량)
    dis_rows = filtered_df[filtered_df['type'] == 'discharge']
    first_cap = dis_rows['Capacity'].iloc[0] if not dis_rows.empty else None

    for _, row in filtered_df.iterrows():
        file_path = os.path.join(data_folder, row['filename'])

        if os.path.exists(file_path):
            temp_df = pd.read_csv(file_path)
            d_type = row['type']
            current_cycle = cycle_counters[d_type]

            # --- [1단계: 파생 변수 로직 정의] ---
            # SOH 정의: 오직 discharge 타입일 때만 해당 사이클의 용량으로 계산
            if d_type == 'discharge' and first_cap and pd.notnull(row['Capacity']):
                soh_val = (row['Capacity'] / first_cap) * 100
            else:
                soh_val = np.nan # 다른 타입은 우선 NaN 처리
            
            # EOL 정의: 미리 계산된 eol_dict 이용
            eol_val = eol_dict.get(battery_id)
            
            # RUL 정의: (사망 사이클 - 현재 사이클)
            rul_val = (eol_val - current_cycle) if pd.notnull(eol_val) else np.nan
            if pd.notnull(rul_val): rul_val = max(0, rul_val)

            # --- [2단계: 데이터프레임 열 추가] ---
            temp_df['start_time'] = row['start_time']
            temp_df['battery_id'] = battery_id
            temp_df['type'] = d_type
            temp_df['ambient_temperature'] = row['ambient_temperature']
            
            temp_df['cycle'] = current_cycle
            temp_df['SOH'] = soh_val  # 정의된 값 주입
            temp_df['EOL_cycle'] = eol_val
            temp_df['RUL'] = rul_val

            # 타입별 특화 데이터 추가
            if d_type == 'discharge':
                temp_df['Capacity'] = row['Capacity']
            elif d_type == 'impedance':
                temp_df['Re'] = row['Re']
                temp_df['Rct'] = row['Rct']

            # 카운터 증가 및 저장
            cycle_counters[d_type] += 1
            key = f"df_{d_type}_{battery_id}"
            if key not in collected_data:
                collected_data[key] = []
            collected_data[key].append(temp_df)

# --- [3단계: 데이터 통합 및 결측치 전파] ---
for key, df_list in collected_data.items():
    combined_df = pd.concat(df_list, ignore_index=True)
    
    # SOH 전파: discharge 파일들 사이의 간극(charge, impedance 등)을 메워줌
    # 단, 사용자님이 'discharge에만' 있길 원하신다면 이 섹션을 제외하면 됩니다.
    # 만약 모든 행에 SOH가 필요하다면 아래 ffill/bfill을 유지하세요.
    if 'SOH' in combined_df.columns:
        combined_df['SOH'] = combined_df['SOH'].ffill()
        combined_df['SOH'] = combined_df['SOH'].bfill()
    
    globals()[key] = combined_df
    print(f"✅ {key} 생성 완료 | 파일 {len(df_list)}개 통합 | 크기: {combined_df.shape} | SOH 결측치: {combined_df['SOH'].isnull().sum()} | cycle/SOH/EOL/RUL 파생 완료")


✅ df_charge_B0005 생성 완료 | 파일 170개 통합 | 크기: (541173, 14) | SOH 결측치: 541173 | cycle/SOH/EOL/RUL 파생 완료
✅ df_discharge_B0005 생성 완료 | 파일 168개 통합 | 크기: (50285, 15) | SOH 결측치: 0 | cycle/SOH/EOL/RUL 파생 완료
✅ df_impedance_B0005 생성 완료 | 파일 278개 통합 | 크기: (13344, 15) | SOH 결측치: 13344 | cycle/SOH/EOL/RUL 파생 완료
✅ df_charge_B0006 생성 완료 | 파일 170개 통합 | 크기: (541173, 14) | SOH 결측치: 541173 | cycle/SOH/EOL/RUL 파생 완료
✅ df_discharge_B0006 생성 완료 | 파일 168개 통합 | 크기: (50285, 15) | SOH 결측치: 0 | cycle/SOH/EOL/RUL 파생 완료
✅ df_impedance_B0006 생성 완료 | 파일 278개 통합 | 크기: (13344, 15) | SOH 결측치: 13344 | cycle/SOH/EOL/RUL 파생 완료
✅ df_charge_B0007 생성 완료 | 파일 170개 통합 | 크기: (541173, 14) | SOH 결측치: 541173 | cycle/SOH/EOL/RUL 파생 완료
✅ df_discharge_B0007 생성 완료 | 파일 168개 통합 | 크기: (50285, 15) | SOH 결측치: 0 | cycle/SOH/EOL/RUL 파생 완료
✅ df_impedance_B0007 생성 완료 | 파일 278개 통합 | 크기: (13344, 15) | SOH 결측치: 13344 | cycle/SOH/EOL/RUL 파생 완료
✅ df_charge_B0018 생성 완료 | 파일 134개 통합 | 크기: (279810, 14) | SOH 결측치: 279810 | cycle/SOH/EOL/RUL 파생 완료
✅ df_i

In [5]:
df_impedance_B0005.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13344 entries, 0 to 13343
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Sense_current        13344 non-null  object        
 1   Battery_current      13344 non-null  object        
 2   Current_ratio        13344 non-null  object        
 3   Battery_impedance    13344 non-null  object        
 4   Rectified_Impedance  10842 non-null  object        
 5   start_time           13344 non-null  datetime64[ns]
 6   battery_id           13344 non-null  object        
 7   type                 13344 non-null  object        
 8   ambient_temperature  13344 non-null  int64         
 9   cycle                13344 non-null  int64         
 10  SOH                  0 non-null      float64       
 11  EOL_cycle            13344 non-null  int64         
 12  RUL                  13344 non-null  int64         
 13  Re                   13344 non-

In [ ]:
df_discharge_B0005

In [ ]:
# 간단 확인
display(df_charge_B0006)
display(df_discharge_B0007)
display(df_impedance_B0005) 

In [ ]:
# 분석가 팁: 반복되는 작업을 줄이기 위해 리스트를 순회하며 출력합니다.

# 1. 방전(Discharge) 데이터 확인
print("=== [DISCHARGE DATA SHAPE & PREVIEW] ===")
discharge_dfs = [df_discharge_B0005, df_discharge_B0006, df_discharge_B0007, df_discharge_B0018]
df_names_dis = ['B0005', 'B0006', 'B0007', 'B0018']

for name, df_obj in zip(df_names_dis, discharge_dfs):
    print(f"Discharge_{name} Shape: {df_obj.shape}")
    display(df_obj.head())  # 전체 출력 대신 head()를 권장하지만, 원하시면 그대로 사용 가능합니다.
    print("-" * 50)

# 2. 임피던스(Impedance) 데이터 확인
print("\n=== [IMPEDANCE DATA SHAPE & PREVIEW] ===")
impedance_dfs = [df_impedance_B0005, df_impedance_B0006, df_impedance_B0007, df_impedance_B0018]
df_names_imp = ['B0005', 'B0006', 'B0007', 'B0018']

for name, df_obj in zip(df_names_imp, impedance_dfs):
    print(f"Impedance_{name} Shape: {df_obj.shape}")
    display(df_obj.head())
    print("-" * 50)

# 3. 충전(Charge) 데이터 확인
print("\n=== [CHARGE DATA SHAPE & PREVIEW] ===")
charge_dfs = [df_charge_B0005, df_charge_B0006, df_charge_B0007, df_charge_B0018]
df_names_cha = ['B0005', 'B0006', 'B0007', 'B0018']

for name, df_obj in zip(df_names_cha, charge_dfs):
    print(f"Charge_{name} Shape: {df_obj.shape}")
    display(df_obj.head())
    print("-" * 50)

# 분석

# discharge 분석

배터리 4개를 모두 합침

각 배터리의 초기의 첫Ah과 최종의 첫Ah를 확인 (떨어짐 확인)

Capacity 값을 그래프로 확인

각 배터리의 1cycle의 Ah값을 시간으로 적분한 값과 capacity를 비교 (거의 같음)

In [ ]:
# 1. 4개 배터리 데이터를 하나로 통합 (df_battery_discharge)

battery_ids = ['B0005', 'B0006', 'B0007', 'B0018']
all_dfs = []

for b_id in battery_ids:
    df_name = f"df_discharge_{b_id}"
    if df_name in globals():
        # .copy()를 써서 원본 데이터와 연결을 끊고 독립적인 복사본을 만들기
        temp_df = globals()[df_name].copy()
        temp_df['battery_id'] = b_id  # 배터리 식별자 추가
        all_dfs.append(temp_df)

# 통합 데이터프레임 생성
df_battery_discharge = pd.concat(all_dfs, ignore_index=True)

print(f"✅ 통합 완료! 전체 행 개수: {len(df_battery_discharge)}")
print(f"체크 - 포함된 배터리들: {df_battery_discharge['battery_id'].unique()}")

In [ ]:
# 배터리별 초기/끝 Capacity 값 

# 1. 배터리별 첫 번째(초기)와 마지막(최종) 사이클의 용량 정보 추출
# groupby와 agg를 사용하여 필요한 값만 한 번에 가져옵니다.
summary_df = df_battery_discharge.groupby('battery_id').agg(
    Initial_Capacity=('Capacity', 'first'),
    Final_Capacity=('Capacity', 'last'),
    Total_Cycles=('cycle_index', 'max')
).reset_index()

# 2. 감소율(Degradation Rate) 계산
summary_df['Degradation_Rate (%)'] = (
    (summary_df['Initial_Capacity'] - summary_df['Final_Capacity']) / 
    summary_df['Initial_Capacity'] * 100
).round(2)

# 3. 소수점 정리 및 결과 출력
summary_df['Initial_Capacity'] = summary_df['Initial_Capacity'].round(4)
summary_df['Final_Capacity'] = summary_df['Final_Capacity'].round(4)

display(summary_df)

In [ ]:
# Capacity 시각화
plt.figure(figsize=(12, 6))

# Seaborn을 사용하면 통합 데이터프레임에서 hue 옵션 하나로 간단히 구분 가능합니다.
sns.lineplot(data=df_battery_discharge, x='cycle_index', y='Capacity', hue='battery_id', marker='o', markersize=4)

# 수명 종료 선 (EOL: End of Life) 설정
plt.axhline(y=1.4, color='red', linestyle='--', label='EOL Threshold (1.4Ah)')

plt.title('Lithium-ion Battery Capacity Fade Trend (Integrated Data)', fontsize=14)
plt.xlabel('Cycle Index')
plt.ylabel('Capacity (Ah)')
plt.legend(title='Battery ID')
plt.grid(True, linestyle=':', alpha=0.7)
plt.show()

# 3. (선택사항) 통합 데이터 CSV 저장
# df_battery_discharge.to_csv("df_battery_discharge.csv", index=False)

In [ ]:
# discharge 배터리별 전류적분값과 capacity값을 비교

def calculate_metrics(group):
    # 각 그룹 내에서 시간 차이 계산
    # 'Time' 컬럼이 있는 통합 데이터라면 바로 작동합니다.
    dt = group['Time'].diff().fillna(0)
    
    # 전류 적분 (Ah) 계산
    integrated_ah = (group['Current_measured'].abs() * dt).sum() / 3600
    
    # 해당 사이클의 Capacity 값 (첫 행)
    provided_cap = group['Capacity'].iloc[0]
    
    return pd.Series({
        'Integrated_Ah': integrated_ah,
        'Provided_Capacity': provided_cap
    })

# 배터리 ID와 사이클 인덱스로 그룹화하여 결과 도출
# (이미 배터리 ID가 컬럼으로 들어있다고 가정합니다)
final_comparison_df = (
    df_battery_discharge
    .groupby(['battery_id', 'cycle_index'])
    .apply(calculate_metrics)
    .reset_index()
)

# 오차율 계산
final_comparison_df['Error_Rate(%)'] = (
    abs(final_comparison_df['Integrated_Ah'] - final_comparison_df['Provided_Capacity']) 
    / final_comparison_df['Provided_Capacity'] * 100
)

display(final_comparison_df)

In [ ]:
# 사이클별 전류적분값과 capacity값 시각화

# Seaborn의 FacetGrid를 사용하면 배터리별 격자(2x2)를 자동으로 생성
g = sns.FacetGrid(final_comparison_df, col="battery_id", col_wrap=2, height=4, aspect=1.5)

# 1. NASA 제공 capacity 값 (실선)
g.map(sns.lineplot, "cycle_index", "Provided_Capacity", color='royalblue', label='NASA Provided', alpha=0.7)

# 2. 직접 적분한 값 (산점도)
g.map(sns.scatterplot, "cycle_index", "Integrated_Ah", color='darkorange', label='My Integrated', s=10, alpha=0.5)

# 3. 디자인 디테일 설정
g.set_axis_labels("Cycle Index", "Capacity (Ah)")
g.add_legend()
g.set_titles("Battery {col_name}: Capacity Comparison", size=12, weight='bold')

plt.subplots_adjust(top=0.9)
g.fig.suptitle('Verification: Provided Capacity vs. Current Integration (Ah)', fontsize=16)
plt.show()

In [ ]:
# 더 볼것없을까...?...방전에 있는 컬럼이 이건데.. 
# 'Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_load', 'Voltage_load', 'Time', 'type', 'start_time', 
# 'ambient_temperature', 'battery_id', 'cycle_index', 'Capacity' 
# 이중에서 일단 Current_measured, capacity 이거를 사용했어 이제 뭘 봐야할까?
# 1. 전압강하 특성 분석, 2. 온도변화와 용량의 상관관계, 3. 방전 부하의 안정성 확인

In [ ]:
# 각 배터리별 방전 데이터의 'Time' 평균값을 비교 (완전히 똑같으면 복제된 데이터임)
for b_id in ['B0005', 'B0006', 'B0007', 'B0018']:
    df_name = f"df_discharge_{b_id}"
    if df_name in globals():
        mean_time = globals()[df_name]['Time'].mean()
        max_time = globals()[df_name]['Time'].max()
        min_time = globals()[df_name]['Time'].min()
        print(f"[{b_id}] Time 평균: {mean_time:.6f} | 최대값: {max_time:.6f} | 최솟값: {min_time:.6f}")

In [ ]:
# 부하가 일정했는지 확인해보는 코드
print(df_battery_discharge[['Current_load', 'Voltage_load','ambient_temperature']].nunique())
print("\n")
print(df_battery_discharge[['Current_load', 'Voltage_load']].describe())
# 2A정도로 통제 / 전압값은 4.249부터 0까지 떨어졌나봄

In [ ]:
# 각 배터리가 참조하는 파일 이름들이 정말 고유한지 확인
for b_id in ['B0005', 'B0006', 'B0007', 'B0018']:
    unique_files = df[df['battery_id'] == b_id]['filename'].nunique()
    first_file = df[df['battery_id'] == b_id]['filename'].iloc[0] if unique_files > 0 else "None"
    print(f"배터리 {b_id}: 고유 파일 개수 = {unique_files}개 / 첫 번째 파일명 = {first_file}")

In [ ]:
# 각 배터리가 참조하는 파일 이름들이 정말 고유한지 확인
for b_id in ['B0005', 'B0006', 'B0007', 'B0018']:
    unique_files = df[df['battery_id'] == b_id]['filename'].nunique()
    first_file = df[df['battery_id'] == b_id]['filename'].iloc[1] if unique_files > 0 else "None"
    print(f"배터리 {b_id}: 고유 파일 개수 = {unique_files}개 / 첫 번째 파일명 = {first_file}")

In [ ]:
# 각 배터리별 첫 번째 사이클의 Duration 값을 출력
for b_id in ['B0005', 'B0006', 'B0007', 'B0018']:
    val = df_plot[(df_plot['battery_id'] == b_id) & (df_plot['cycle_index'] == 1)]['Duration'].values
    print(f"{b_id}의 1번 사이클 Duration: {val}")

In [ ]:

# 1. 2x2 서브플롯 생성
fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharex=True)
axes = axes.flatten()
target_ids = ['B0005', 'B0006', 'B0007', 'B0018']

for i, b_id in enumerate(target_ids):
    # 해당 배터리 데이터만 딱 집어내기
    target_data = df_plot[df_plot['battery_id'] == b_id]
    
    # 만약 데이터가 비어있다면 출력
    if target_data.empty:
        axes[i].text(0.5, 0.5, f'No Data for {b_id}', ha='center')
        continue
        
    sns.lineplot(data=target_data, x='cycle_index', y='Duration', ax=axes[i], color='red')
    axes[i].set_title(f'Battery {b_id} Discharge Duration')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 계산된 방전 시간의 요약 통계 확인

# 1. df_plot 정의 (에러 해결 핵심 구간)
# 배터리ID와 사이클별로 Time의 최대값에서 최소값을 빼서 '방전 지속 시간'을 구합니다.
df_plot = df_battery_discharge.groupby(['battery_id', 'cycle_index'])['Time'].agg(lambda x: x.max() - x.min()).reset_index()
df_plot.columns = ['battery_id', 'cycle_index', 'Duration']

# 2. 통계 확인 
print("배터리별 계산된 방전 시간(Duration) 통계:")
summary = df_plot.groupby('battery_id')['Duration'].agg(['min', 'max', 'mean', 'std'])
print(summary)

# 시각화를 위해 억지로 1초씩 차이를 둬서 선이 3개인지 확인만 해보는 코드
plt.figure(figsize=(12, 6))
df_plot_offset = df_plot.copy()
df_plot_offset.loc[df_plot_offset['battery_id']=='B0006', 'Duration'] += 10 
df_plot_offset.loc[df_plot_offset['battery_id']=='B0007', 'Duration'] += 20

sns.lineplot(data=df_plot_offset, x='cycle_index', y='Duration', hue='battery_id')
plt.title("Forced Separation to Confirm Data Presence")
plt.show()

In [ ]:


# 1. 방전 시간 계산 (최대 - 최소 시간)
discharge_duration = (
    df_battery_discharge
    .groupby(['battery_id', 'cycle_index'])['Time']
    .agg(lambda x: x.max() - x.min())
    .reset_index()
)
discharge_duration.columns = ['battery_id', 'cycle_index', 'Duration_Sec']

# 2. 4분할 그래프 그리기 (2x2)
fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharex=True)
axes = axes.flatten() # 2D 배열을 1D로 펴서 반복문 돌리기 편하게 함
target_ids = ['B0005', 'B0006', 'B0007', 'B0018']
colors = ['red', 'blue', 'green', 'orange']

for i, b_id in enumerate(target_ids):
    # 해당 배터리 데이터만 필터링
    data = discharge_duration[discharge_duration['battery_id'] == b_id]
    
    if not data.empty:
        sns.lineplot(data=data, x='cycle_index', y='Duration_Sec', 
                     ax=axes[i], color=colors[i], marker='o', markersize=2)
        axes[i].set_title(f'Battery {b_id} - Discharge Duration', fontsize=12)
        axes[i].grid(True, linestyle=':', alpha=0.6)
    else:
        axes[i].set_title(f'Battery {b_id} - NO DATA FOUND', color='red')

plt.tight_layout()
plt.show()

# 3. 데이터가 정말 몇 개씩 있는지 숫자로 찍어보기 (확인 사살)
print("각 배터리별 사이클 데이터 개수:")
print(discharge_duration['battery_id'].value_counts())

In [ ]:
# 사이클별 방전 시간(초) 계산
discharge_duration = df_battery_discharge.groupby(['battery_id', 'cycle_index'])['Time'].max().reset_index()

plt.figure(figsize=(10, 6))
sns.lineplot(data=discharge_duration, x='cycle_index', y='Time', hue='battery_id')
plt.title('Discharge Duration per Cycle (Total Seconds)')
plt.ylabel('Time (s)')
plt.grid(True)
plt.show()

In [ ]:

# 1. 배터리별/사이클별 시작 및 종료 전압 추출
voltage_analysis = df_battery_discharge.groupby(['battery_id', 'cycle_index'])['Voltage_measured'].agg(['first', 'last']).reset_index()

# 컬럼명 변경 (시작 전압, 종료 전압)
voltage_analysis.columns = ['battery_id', 'cycle_index', 'V_start', 'V_end']

# 2. 전압 차이(Drop) 계산
# 차이가 클수록 해당 시간 동안 에너지를 많이 소모했거나 저항이 커서 전압이 빨리 떨어진 것임
voltage_analysis['V_drop'] = voltage_analysis['V_start'] - voltage_analysis['V_end']

# 3. 배터리별 평균 전압 하락폭 통계 확인
print("📊 [배터리별 전압 분석 요약]")
summary = voltage_analysis.groupby('battery_id')[['V_start', 'V_end', 'V_drop']].min()
print(summary)

# 4. 특정 사이클(예: 1번 사이클) 결과만 상세 보기
print("\n🔍 [1번 사이클 상세 비교]")
print(voltage_analysis[voltage_analysis['cycle_index'] == 1])

In [ ]:
plt.figure(figsize=(12, 6))
# 사이클이 진행될수록 V_drop이 커지는지(즉, 전압이 더 빨리 떨어지는지) 확인
sns.lineplot(data=voltage_analysis, x='cycle_index', y='V_drop', hue='battery_id', palette='viridis')

plt.title('How much voltage drops during the same period', fontsize=15)
plt.axhline(y=1.5, color='r', linestyle='--', label='Danger Zone (High Drop)') # 임의의 기준선
plt.ylabel('Voltage Drop (V)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# 동일한 사이클(예: 50번 사이클)에서 종료 직전 전압 비교
target_cycle = 50 

print(f"📊 [사이클 {target_cycle}] 종료 직전(3690초 부근) 전압 비교")
for b_id in ['B0005', 'B0006', 'B0007', 'B0018']:
    # 해당 배터리/사이클의 마지막 행 가져오기
    last_row = df_battery_discharge[(df_battery_discharge['battery_id'] == b_id) & 
                                     (df_battery_discharge['cycle_index'] == target_cycle)].iloc[-1:]
    
    if not last_row.empty:
        v_end = last_row['Voltage_measured'].values[0]
        t_end = last_row['Time'].values[0]
        print(f"{b_id} -> 종료시간: {t_end:.1f}s | 종료전압: {v_end:.4f}V")

In [ ]:
# 2. 온도변화와 용량의 상관관계

# 1. 사이클별 평균 온도 추출
temp_summary = df_battery_discharge.groupby(['battery_id', 'cycle_index'])['Temperature_measured'].mean().reset_index()

# 2. 기존 비교 데이터프레임과 합치기
analysis_df = pd.merge(final_comparison_df, temp_summary, on=['battery_id', 'cycle_index'])

# 3. 시각화 (용량 감소와 온도 상승의 관계)
fig, ax1 = plt.subplots(figsize=(12, 6))

# 용량 그래프 (왼쪽 축)
sns.lineplot(data=analysis_df, x='cycle_index', y='Provided_Capacity', hue='battery_id', ax=ax1, legend=False)
ax1.set_ylabel('Capacity (Ah)', color='blue')

# 온도 그래프 (오른쪽 축)
ax2 = ax1.twinx()
sns.lineplot(data=analysis_df, x='cycle_index', y='Temperature_measured', hue='battery_id', ax=ax2, palette='flare', alpha=0.3)
ax2.set_ylabel('Avg Temperature (°C)', color='red')

plt.title('Capacity Fade vs. Temperature Trend')
plt.show()

In [ ]:
# cycle별 다양한 시각화

# 데이터가 많으므로 방전 시간 전처리를 위해 그룹화
def precompute_discharge_time(group):
    # 방전 지속 시간 (분 단위)
    discharge_time_min = (group['Time'].max() - group['Time'].min()) / 60
    # 평균 온도
    avg_temp = group['Temperature_measured'].mean()
    # 요약 데이터 반환
    return pd.Series({
        'Discharge_Time_Min': discharge_time_min,
        'Avg_Temperature': avg_temp,
        # 용량은 사이클당 하나이므로 첫 값 사용
        'Capacity': group['Capacity'].iloc[0]
    })

# 전처리 실행 (배터리별, 사이클별)
summary_df = df_battery_discharge.groupby(['battery_id', 'cycle_index']).apply(precompute_discharge_time).reset_index()

# 시각화 설정
fig, axes = plt.subplots(2, 2, figsize=(18, 12), gridspec_kw={'hspace': 0.3, 'wspace': 0.2})
plt.suptitle('Lithium-ion Battery Multi-dimensional Degradation Analysis', fontsize=20, y=0.95)

# --- [1. 용량 감소 추세 (Capacity Fade)] ---
ax1 = axes[0, 0]
sns.lineplot(data=summary_df, x='cycle_index', y='Capacity', hue='battery_id', marker='o', markersize=3, ax=ax1)
ax1.axhline(y=1.4, color='red', linestyle='--', label='EOL Threshold (1.4Ah)')
ax1.set_title('1. Capacity Fade Trend', fontsize=14)
ax1.set_ylabel('Capacity (Ah)')
ax1.set_xlabel('Cycle Index')
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(title='Battery ID')

# --- [2. 전압 강하 패턴 비교 (Voltage Profile)] ---
ax2 = axes[0, 1]
# 비교를 위해 특정 배터리(예: B0005)의 초기와 말기 사이클 추출
target_b_id = 'B0018'
cycles_to_compare = [1, 80] # 데이터에 존재하는 사이클 번호로 조정 필요

# 전체 통합 데이터에서 원본 시계열 추출
v_profile_df = df_battery_discharge[
    (df_battery_discharge['battery_id'] == target_b_id) & 
    (df_battery_discharge['cycle_index'].isin(cycles_to_compare))
].copy()

# 시각화를 위해 사이클 이름을 문자열로 변경
v_profile_df['Cycle'] = v_profile_df['cycle_index'].astype(str)

sns.lineplot(data=v_profile_df, x='Time', y='Voltage_measured', hue='Cycle', palette='coolwarm', ax=ax2)
ax2.set_title(f'2. Voltage Profile (Battery {target_b_id}: Early vs Late Cycle)', fontsize=14)
ax2.set_ylabel('Voltage (V)')
ax2.set_xlabel('Discharge Time (s)')
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(title='Cycle Index')

# --- [3. 평균 온도 변화 (Avg Temperature)] ---
ax3 = axes[1, 0]
sns.lineplot(data=summary_df, x='cycle_index', y='Avg_Temperature', hue='battery_id', marker='s', markersize=3, alpha=0.6, ax=ax3)
ax3.set_title('3. Average Temperature Trend per Cycle', fontsize=14)
ax3.set_ylabel('Avg Temperature (°C)')
ax3.set_xlabel('Cycle Index')
ax3.grid(True, linestyle=':', alpha=0.6)
ax3.legend(title='Battery ID', loc='upper left')

# --- [4. 방전 지속 시간 단축 (Discharge Time)] ---
ax4 = axes[1, 1]
sns.lineplot(data=summary_df, x='cycle_index', y='Discharge_Time_Min', hue='battery_id', marker='^', markersize=3, ax=ax4)
ax4.set_title('4. Discharge Duration Shortening', fontsize=14)
ax4.set_ylabel('Duration (Minutes)')
ax4.set_xlabel('Cycle Index')
ax4.grid(True, linestyle=':', alpha=0.6)
ax4.legend(title='Battery ID')

print("시각화 완료.")
plt.show()

In [ ]:


# 1. 분석할 배터리 선택 및 비교할 사이클 설정
target_b_id = 'B0005'
# 초기(10), 중기(80), 말기(150) 사이클을 선택하여 비교
comparison_cycles = [10, 80, 150] 

# 2. 데이터 필터링
# 통합 데이터프레임에서 해당 배터리와 선택한 사이클만 추출
v_trend_df = df_battery_discharge[
    (df_battery_discharge['battery_id'] == target_b_id) & 
    (df_battery_discharge['cycle_index'].isin(comparison_cycles))
].copy()

# 시각화를 위해 사이클 번호를 문자열(범주형)로 변환
v_trend_df['Cycle'] = v_trend_df['cycle_index'].apply(lambda x: f"Cycle {x}")

# 3. 시각화
plt.figure(figsize=(12, 7))

# 시간(Time)에 따른 전압(Voltage_measured) 변화
sns.lineplot(data=v_trend_df, x='Time', y='Voltage_measured', hue='Cycle', palette='viridis', linewidth=2)

# 방전 종지 전압(Cut-off Voltage) 표시 (보통 2.7V ~ 3.0V 사이)
plt.axhline(y=2.7, color='red', linestyle='--', label='Cut-off Voltage (2.7V)')

plt.title(f'Voltage Discharge Profile by Cycle - Battery {target_b_id}', fontsize=15)
plt.xlabel('Discharge Time (seconds)', fontsize=12)
plt.ylabel('Measured Voltage (V)', fontsize=12)
plt.legend(title="Degradation Stage")
plt.grid(True, linestyle=':', alpha=0.7)

plt.show()

In [ ]:
df_battery_discharge.columns

In [ ]:
# soh

# 1. 배터리 ID와 사이클 번호로 정렬 (시간 순서 보장)
df_battery_discharge = df_battery_discharge.sort_values(by=['battery_id', 'cycle_index'])

# 2. 각 배터리(battery_id) 그룹별로 '맨 처음(첫 행)'의 Capacity를 찾아 새로운 컬럼으로 생성
# transform('first')를 쓰면 그룹 내 첫 번째 값이 모든 사이클 행에 분모로 들어갑니다.
df_battery_discharge['initial_capacity'] = df_battery_discharge.groupby('battery_id')['Capacity'].transform('first')

# 3. SOH 계산: (현재 사이클의 capacity / 해당 배터리의 초기 capacity) * 100
df_battery_discharge['SOH'] = (df_battery_discharge['Capacity'] / df_battery_discharge['initial_capacity']) * 100

# 4. 결과 확인 (배터리 ID, 사이클, 현재용량, 초기용량, SOH 순서)
print(df_battery_discharge[['battery_id', 'cycle_index', 'Capacity', 'initial_capacity', 'SOH']])

In [ ]:
# 1. 시각화 스타일 설정 (Seaborn 기본 테마)
sns.set_theme(style="whitegrid")

# 2. FacetGrid 생성: battery_id 컬럼을 기준으로 격자(Grid) 나누기
# col="battery_id": 배터리 ID별로 가로로 나열 (칸)
# col_wrap=2: 한 줄에 최대 2개씩 배치 (배터리가 많을 때 유용)
# sharex=False, sharey=True: X축(사이클) 범위는 달라도 되지만, Y축(SOH) 범위는 통일
g = sns.FacetGrid(df_battery_discharge, col="battery_id", col_wrap=2, 
                  height=4, aspect=1.2, sharex=False, sharey=True)

# 3. 각 격자에 실제 그래프 그리기 (꺾은선 그래프 + 산점도)
# sns.lineplot: 꺾은선
# "cycle_index": X축
# "SOH": Y축
g.map(sns.lineplot, "cycle_index", "SOH", marker="o", color="rebeccapurple", linewidth=1.5, markersize=4)

# 4. 그래프 디테일 설정
# 각 서브플롯의 제목 형식 설정 (예: Battery: B0005)
g.set_titles(template="Battery: {col_name}")

# X축, Y축 라벨 설정
g.set_axis_labels("Cycle Index", "SOH (%)")

# Y축 범위 고정 (SOH는 0~100% 사이이므로)
# 데이터에 따라 100%를 약간 넘을 수 있다면 범위를 조절하세요 (예: 0, 110)
g.set(ylim=(0, 105)) 

# 5. 수명 종료(EOL) 기준선 추가 (옵션: SOH 80%)
# 각 서브플롯을 순회하며 빨간 점선 추가
for ax in g.axes.flat:
    ax.axhline(y=80, color='r', linestyle='--', linewidth=1, label='EOL (80%)')
    # 첫 번째 그래프에만 범례 표시 (깔끔하게)
    if ax == g.axes.flat[0]:
        ax.legend(loc='lower left', fontsize='small')

# 6. 레이아웃 조정 및 표시
plt.tight_layout()
plt.show()

# 충전 분석

In [ ]:
# 1. 충전(Charge) 데이터 통합
battery_ids = ['B0005', 'B0006', 'B0007', 'B0018']
all_charge_dfs = []

for b_id in battery_ids:
    df_name = f"df_charge_{b_id}"
    if df_name in globals():
        # 복사본 생성 및 ID 추가
        temp_df = globals()[df_name].copy()
        temp_df['battery_id'] = b_id
        all_charge_dfs.append(temp_df)

# 통합 데이터프레임 생성
df_battery_charge = pd.concat(all_charge_dfs, ignore_index=True)

print(f"charge 통합 - 전체 행 개수: {len(df_battery_charge)}")
print(f"포함된 배터리 id: {df_battery_charge['battery_id'].unique()}")

# 임피던스 분석





In [ ]:
# 임피던스(Impedance) 데이터 통합
battery_ids = ['B0005', 'B0006', 'B0007', 'B0018']
all_impedance_dfs = []

for b_id in battery_ids:
    df_name = f"df_impedance_{b_id}"
    if df_name in globals():
        # 복사본 생성 및 ID 추가
        temp_df = globals()[df_name].copy()
        temp_df['battery_id'] = b_id
        all_impedance_dfs.append(temp_df)

# 통합 데이터프레임 생성
df_battery_impedance = pd.concat(all_impedance_dfs, ignore_index=True)

print(f"Impedance 통합 - 전체 행 개수: {len(df_battery_impedance)}")
print(f"포함된 배터리 id: {df_battery_impedance['battery_id'].unique()}")

In [ ]:
# 임피던스 배터리 모두 통합 (df_battery_impedance)

target_ids = ['B0005', 'B0006', 'B0007', 'B0018']
master_list = []

for b_id in target_ids:
    # 1. 기존 방전 요약 데이터에서 해당 배터리 추출
    dis_data = final_comparison_df[final_comparison_df['battery_id'] == b_id].copy()
    
    # 2. 이미 존재하는 임피던스 변수 가져오기 (globals 활용)
    imp_df_name = f"df_impedance_{b_id}"
    if imp_df_name in globals():
        imp_data = globals()[imp_df_name].copy()
        
        # 3. 데이터 결합 (Cycle_index 기준)
        # 모든 컬럼(Sense_current, Rectified_Impedance, Re, Rct 등)을 가져옵니다.
        merged = pd.merge(
            dis_data, 
            imp_data, 
            on=['cycle_index', 'battery_id'], 
            how='outer' # 양쪽 데이터가 모두 보존되도록 outer join
        )
        master_list.append(merged)

# 4. 전체 배터리 통합 마스터 데이터프레임 생성
df_battery_impedance = pd.concat(master_list, ignore_index=True)

# 5. 시계열 순서로 정렬
df_battery_impedance = df_battery_impedance.sort_values(['battery_id', 'cycle_index'])

# 결과 확인
print("✅ 마스터 데이터 통합 완료!")
display(df_battery_impedance)

In [ ]:
# 핵심 시각화 1: 용량 vs 보정된 임피던스 (X자형 교차 그래프)

# [추가] 데이터 타입 클리닝: Rectified_Impedance가 float인지 확인하고 강제 변환
# 만약 리스트나 문자열로 되어 있다면 첫 번째 요소만 추출하거나 숫자로 바꿉니다.
def clean_impedance_value(val):
    try:
        if isinstance(val, (list, np.ndarray)):
            return float(val[0]) # 리스트면 첫 번째 값 추출
        return float(val) # 숫자형태면 그대로 float 변환
    except:
        return np.nan # 변환 실패 시 NaN 처리

# 클리닝 적용
df_battery_impedance['Rectified_Impedance'] = df_battery_impedance['Rectified_Impedance'].apply(clean_impedance_value)
df_battery_impedance['Provided_Capacity'] = pd.to_numeric(df_battery_impedance['Provided_Capacity'], errors='coerce')

# 1. 시각화 스타일 및 한글 폰트 설정 (필요시)
plt.figure(figsize=(18, 12))
sns.set_style("whitegrid")

# 2. 배터리별로 서브플롯 생성 (2x2 형태)
target_ids = ['B0005', 'B0006', 'B0007', 'B0018']
for i, b_id in enumerate(target_ids, 1):
    ax1 = plt.subplot(2, 2, i)
    
    # 해당 배터리 데이터만 필터링 및 정렬
    target_df = df_battery_impedance[df_battery_impedance['battery_id'] == b_id].sort_values('cycle_index')
    
    # --- 왼쪽 Y축: 방전 용량 (Capacity) ---
    color_cap = 'royalblue'
    ax1.set_xlabel('Cycle Index', fontsize=12)
    ax1.set_ylabel('Discharge Capacity (Ah)', color=color_cap, fontsize=12)
    lns1 = ax1.plot(target_df['cycle_index'], target_df['Provided_Capacity'], 
                    label='Capacity (Ah)', color=color_cap, linewidth=2.5, alpha=0.8)
    ax1.tick_params(axis='y', labelcolor=color_cap)
    
    # 수명 종료(EOL) 기준선 추가 (1.4Ah)
    ax1.axhline(y=1.4, color='red', linestyle='--', linewidth=1.5, label='EOL (1.4Ah)')

    # --- 오른쪽 Y축 (이중 축): 보정된 임피던스 (Rectified_Impedance) ---
    ax2 = ax1.twinx()  # 이중 축 설정
    color_imp = 'darkorange'
    ax2.set_ylabel('Rectified Impedance (Ohm)', color=color_imp, fontsize=12)
    # 임피던스는 모든 사이클에 데이터가 없으므로 점(Scatter)으로 표현하는 것이 좋습니다.
    lns2 = ax2.scatter(target_df['cycle_index'], target_df['Rectified_Impedance'], 
                       label='Rectified Impedance', color=color_imp, s=15, alpha=0.6)
    ax2.tick_params(axis='y', labelcolor=color_imp)

    # 타이틀 설정
    plt.title(f'Battery {b_id}: Capacity Fade vs. Impedance Growth', fontsize=14, fontweight='bold')
    
    # 범례 합치기
    lns = lns1 + [lns2]
    labs = [l.get_label() for l in lns]
    # ax1.legend(lns, labs, loc='center right') # 범례 위치 조정 필요시

# 3. 전체 레이아웃 조정 및 출력
plt.tight_layout()
plt.suptitle('Correlation between Capacity Degradation and Impedance Increase', fontsize=18, y=1.02, fontweight='bold')
plt.show()

In [ ]:
# 핵심 시각화 2: 전해질 저항(Re) vs 전하 전달 저항(Rct) 흐름 비교

plt.figure(figsize=(18, 12))

for i, b_id in enumerate(target_ids, 1):
    plt.subplot(2, 2, i)
    
    target_df = df_battery_impedance[df_battery_impedance['battery_id'] == b_id].sort_values('cycle_index')
    
    # Re (Electrolyte Resistance) - 전해질 저항
    plt.plot(target_df['cycle_index'], target_df['Re'], 
             label='Re (Electrolyte)', color='seagreen', linewidth=2, marker='o', markersize=4, alpha=0.7)
    
    # Rct (Charge Transfer Resistance) - 전하 전달 저항
    plt.plot(target_df['cycle_index'], target_df['Rct'], 
             label='Rct (Charge Transfer)', color='crimson', linewidth=2, marker='s', markersize=4, alpha=0.7)
    
    plt.title(f'Battery {b_id}: Impedance Parameter Trend (Re & Rct)', fontsize=14, fontweight='bold')
    plt.xlabel('Cycle Index')
    plt.ylabel('Resistance (Ohm)')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.suptitle('Comparison of Electrolyte (Re) and Charge Transfer (Rct) Resistance Growth', fontsize=18, y=1.02, fontweight='bold')
plt.show()